# Mathematical Reasoning Abilities of Hugging Face LLM Agents

Large Language Models (LLMs) have astounded us with their abilities in performing tasks once considered too creative for artificial agents, such as writing essays or code, solving mathematical problems, and what not. But how good is their mathematical reasoning? 

Here, we start from the lowest of expectations -- Can they add 2 and 2 together? Specifically, we test several models from the Hugging Face Library (https://huggingface.co/models) and examine how many of them can answer this question: "2+2 = ?".

In this guide, we show how to set up the models from HuggingFace on a consumer laptop.

### List of Models Tested


Tried and succeed:
1. DeepSeek Math 7B Instruct
2. DeepSeek Math 7B RL
3. DeepSeek Coder 1.3B
4. MathStral (Note: Gated model, requires HuggingFace Login, please see below)
5. Galactica-125M

Tried and failed:\
Llama-4-Scout-17B-16E-Instruct: Turned out to be too large for our PC even with compression and offloading.


Note that the gated models need login with access tokens. Token generation is simple, but we need to keep the token to ourselves for security reasons. Please read this overview of steps carefully:
    
1. Create a hugging face account.
2. Then, complete agreements for particular models by going to their respective pages.
3. Create an Access Token --> Suggested option: Fine-Grained with all Read permissions (except billing info) turned on.
4. Copy the hf token (starts with "hf_")\
    CAUTION! DO NOT USE TOKEN IN CODE DIRECTLY.\
    Rather, use the steps we show below: saving it in a text file in your personal computer.\
    CAUTION! DO NOT PUSH THE HF TOKEN TO GITHUB or other repos!

## PC Specifications:
    -- NVIDIA GeForce RTX 3070 Laptop GPU with 8 GB VRAM or display memory
    -- Intel i7-12650H CPU, 2.3 GHz with 10 cores and 16 CPU threads
    -- 32 GB 4800 MHz DDR5 RAM
    -- 1TB SSD
    

To note, Deepseek-Math-7B-Instruct, its RL version, and Mathstral models all require 14 GB VRAM. So, we used (1) 4-bit quantization for compressing and (2) offloading to CPU, if needed after compression. We also tested with offloading to CPU only without compression, which significantly slowed down our experiments, such as 1-1.5 min per question compared to 2-6 sec per question when solely on GPU.



## Pre-requisites

We need to have these packages:

    1. transformer
    2. accelerate -- for GPU memory offloading
    3. BitsandBytes -- usually included in the transformer package.

Please follow the instructions below.

In [ ]:
# Install Hugging Face Transformers library
    # Using pip
# !pip install transformers -q ## -q for quiet output
    # Using Conda
# !conda install conda-forge::transformers
    # Likely to need to use terminal or command prompt for Conda

# For memory offloading
    # Using pip
# !pip install 'accelerate>=0.26.0
    # Using conda
# !conda install conda-forge::accelerate

# Update BitsandBytes (only pip option available) 
# !pip install -U bitsandbytes

## Model Manager Class

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
device = "cuda" if torch.cuda.is_available() else "cpu"
import h5py
import pickle
# import numpy as np
import gc

# login needed for mistralai/Llama-4
# from huggingface_hub import login

# with open("rr_hf_token.txt", "r") as f:
#     token = f.read().strip()
# login(token=token)

import time
import os

# Class for managing multiple models
class ModelManager:
    def __init__(self):
        # Note: Each model would have its own tokenizer. Is that necessary? Experiments indicate yes.
        torch.cuda.empty_cache() ## Clear existing models from Cache while creating a new model manager
        self.models = {}
        self.tokenizers = {}
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def load_model(self, model_name, offload_folder = "offload_weights", use_quantization = False):
        if model_name not in self.models:
            time_start = time.time()
            # Load model and tokenizer
            print(f"Loading model: {model_name}")

            # Way 1: All in GPU and if not available, CPU. No Splitting
            # self.models[model_name] = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)
        
            # Way 2: Use "auto" device. 
                # Would try to use GPU. If memory not sufficient, will split and off-load to CPU
                # Possible con: Increased response time.
            
            if (use_quantization == False):
                print("Loading model WITHOUT QUANTIZATION.\n")
                self.models[model_name] = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    torch_dtype=torch.float16,
                    device_map="auto",  # Auto-offload to CPU/RAM if needed
                    offload_folder=offload_folder,
                    use_safetensors=True
                )

            # Way 3: Compress with Bits and Bytes.
            # Load with 8-bit or 4-bit quantization WITHOUT QLoRA

            else:
                print("Loading model WITH QUANTIZATION.\n")
                quantization_config = BitsAndBytesConfig(load_in_4bit=True) # Set load_in_4bit=True for 4-bit quantization
                self.models[model_name] = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    quantization_config = quantization_config,
                    torch_dtype=torch.float16,
                    device_map="auto",  # Auto-offload to CPU/RAM if needed
                    offload_folder=offload_folder,
                    use_safetensors=True
                )
            
            self.tokenizers[model_name] = AutoTokenizer.from_pretrained(model_name)
            
            print("Model device:", self.models[model_name].device)
            print(f"\tModel loading time = {time.time() - time_start}")
        else:
            print(f"Model {model_name} is already loaded.")

    def generate_answer_for_one_model(self, model_name, input_prompt, max_length=2000, print_model_details = 1):
        print(f"Current model: {model_name}")
        if model_name not in self.models:
            raise ValueError(f"Model {model_name} is not loaded. Please load first using load_model(model_name).")
        
        time_start = time.time()
        current_tokenizer = self.tokenizers[model_name]
        current_model = self.models[model_name]

        # ------------- Tokenize Input ------------------

        input_tokens = current_tokenizer(input_prompt, return_tensors = "pt").to(self.device)
        
        # ----------- Generate Output ---------------

        output_tokens = current_model.generate(
                            # input_ids = input_tokens['input_ids'],  # Required input IDs
                            # attention_mask = input_tokens['attention_mask'],  # Pass attention mask for longer inputs
                            **input_tokens,  # Automatic device alignment
                            # max_new_tokens = 100,
                            max_length = max_length,  # Limit the length of the response
                            do_sample = True,  # Enable sampling for more creative responses
                            pad_token_id = current_tokenizer.eos_token_id  # Handle padding correctly
                        )

        if (print_model_details):
            print("Model Details:\n" + "-"*10)
            print(current_model)
            print("\n" + "-"*100)
        output_to_return = current_tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        
        print(f"Time taken to generate reply = {time.time()- time_start}")
        return f"Reply: \n ------------------\n {output_to_return}\n [END of ANSWER]\n"


## Setting Up Models

In [14]:
# List of models to load
model_list = ["deepseek-ai/deepseek-math-7b-instruct"]
   
                # "deepseek-ai/deepseek-math-7b-instruct"
                # "deepseek-ai/deepseek-math-7b-rl" 
                # "deepseek-ai/deepseek-coder-1.3b-base"
                # "meta-llama/Llama-4-Scout-17B-16E-Instruct"
                # "mistralai/mathstral-7b-v0.1"
                # "facebook/galactica-125m"


manager = ModelManager() # Initialize a manager

In [15]:
# load models in manager
for model_to_use in model_list: manager.load_model(model_to_use, use_quantization = True)

Loading model: deepseek-ai/deepseek-math-7b-instruct
Loading model WITH QUANTIZATION.



Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model device: cuda:0
	Model loading time = 17.31516194343567


In [17]:
# ----> Set prompt here <-----
input_prompt = "I will give you four numbers and you will combine them into an expression using addition, subtraction, multiplication, and division to create 10 as the arithmetic result. Each number must be used exactly and only once. Here is the set: [1,2,3,4]."

# input_prompt = "I will give you four numbers and you will combine them into an expression using subtraction only to create 10 as the arithmetic result. Each number must be used once only. Here is the set: [29, 13, 23, 17] to make 10."

# input_prompt = "2+2-2=?"

# input_prompt = "Explain the play on words in this joke. Here is the joke: 'A priest, an imam, and a rabbit walk into a blood bank. The priest says 'I am a Type A'. The imam says 'I am a Type B'. The rabbit says 'I am a Type O'"

for model_to_use in model_list:
    agent_answer = manager.generate_answer_for_one_model(model_to_use, input_prompt, print_model_details = 0)

    print(agent_answer)

Current model: deepseek-ai/deepseek-math-7b-instruct
Time taken to generate reply = 30.65999698638916
Reply: 
 ------------------
 I will give you four numbers and you will combine them into an expression using addition, subtraction, multiplication, and division to create 10 as the arithmetic result. Each number must be used exactly and only once. Here is the set: [1,2,3,4].

## Combination 1:
2 + 1 => 3
3 * 4 => 12
12 - 3 => 9
9 - 3 => 6
6 / 2 => 3
3 * 3 => 9
9 + 1 => 10

## Combination 2:
3 / 3 => 1
1 + 1 => 2
2 + 4 => 6
6 - 1 => 5
5 * 2 => 10

## Combination 3:
4 - 3 => 1
1 + 1 => 2
2 + 2 => 4
4 * 3 => 12
12 / 12 => 1
1 + 1 => 2
2 + 2 => 4
4 / 2 => 2
2 * 3 => 6
6 / 3 => 2
2 + 4 => 6
6 + 1 => 7
7 - 1 => 6
6 / 2 => 3
3 + 1 => 4
4 + 4 => 8
8 * 2 => 16
16 / 4 => 4
4 / 2 => 2
2 + 2 => 4
4 + 3 => 7
7 * 1 => 7
7 + 1 => 8
8 / 4 => 2
2 - 1 => 1
1 * 1 => 1
1 * 2 => 2
2 - 1 => 1
1 / 1 => 1
1 + 3 =>4
4* 2 => 8
8 / 2 => 4
4 + 1 => 5
5 - 1 => 4
4 + 3 => 7
7 / 3 => 2
2 + 4 => 6
6 - 2=> 4
4 / 2 => 

## A Model WITH QUANTIZATION and OFFLOADING

In [5]:
manager = ModelManager() # Initialize a manager

# load models in manager
for model_to_use in model_list: manager.load_model(model_to_use, use_quantization = True)

Loading model: deepseek-ai/deepseek-math-7b-instruct
Loading model WITH QUANTIZATION.



Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model device: cuda:0
	Model loading time = 14.41390585899353


In [11]:
# ----> Set prompt here <-----
input_prompt = "What is 23+171?"
# input_prompt.to("cuda")


for model_to_use in model_list:
    agent_answer = manager.generate_answer_for_one_model(model_to_use, input_prompt, print_model_details = 0)

    print(agent_answer)

Current model: deepseek-ai/deepseek-math-7b-instruct
Time taken to generate reply = 2.9470419883728027
Reply: 
 ------------------
 What is 23+171?<br>23<br>+171<br>------<br>194<br>-----<br>So, 23 + 171 = <span class="big-number">194</span>
 [END of ANSWER]



## Numberland-Lite

## Useful things to check

In [7]:
manager.models[model_to_use].config.max_position_embeddings

10485760

In [ ]:
# manager.tokenizers[model_to_use]
manager.models[model_to_use].hf_device_map

## Saving Activations

In [ ]:
class ModelManagerExtreme:
    def __init__(self):
        self.models = {}
        self.tokenizers = {}
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.activations = {}

    def load_model(self, model_name, offload_folder="offload_weights"):
        if model_name not in self.models:
            print(f"Loading model: {model_name}")
            os.makedirs(offload_folder, exist_ok=True)
            self.models[model_name] = AutoModelForCausalLM.from_pretrained(
                model_name,
                torch_dtype=torch.float16,
                device_map="auto",
                offload_folder=offload_folder,
                use_safetensors=True
            )
            self.tokenizers[model_name] = AutoTokenizer.from_pretrained(model_name)
            print("Model loaded successfully!")
        else:
            print(f"Model {model_name} is already loaded.")

    def register_hooks(self, model_name, layers_to_hook="all"):
        if model_name not in self.models:
            raise ValueError(f"Model {model_name} is not loaded.")
        
        def hook_fn(name):
            def hook(module, input, output):
                self.activations[name] = output[0].detach() if isinstance(output, tuple) else output.detach()
            return hook

        def detect_model_architecture(model):
            if hasattr(model, "model"):
                layers = model.model.layers
                return layers, "up_proj", "down_proj", "self_attn", "mlp"
            elif hasattr(model, "transformer"):
                layers = model.transformer.h
                return layers, "c_fc", "c_proj", "attn", "mlp"
            else:
                raise ValueError("Unknown architecture")

        model = self.models[model_name]
        layers, ffn_in, ffn_out, attn_attr, ffn_module = detect_model_architecture(model)
        num_layers = len(layers)
        start_layer = 0 if layers_to_hook == "all" else num_layers - 3

        for layer_idx in range(start_layer, num_layers):
            layer = layers[layer_idx]
            getattr(layer, attn_attr).register_forward_hook(hook_fn(f"attn_layer{layer_idx}_{model_name}"))
            getattr(getattr(layer, ffn_module), ffn_in).register_forward_hook(hook_fn(f"ffn_in_layer{layer_idx}_{model_name}"))
            getattr(getattr(layer, ffn_module), ffn_out).register_forward_hook(hook_fn(f"ffn_out_layer{layer_idx}_{model_name}"))

    def save_activations(self, model_name, prompt):
        if not self.activations:
            print(f"No activations to save for {model_name} with prompt: {prompt}")
            return

        print("Saving activations in H5 format.")
        with h5py.File("activations.h5", "a") as f:
            for key, act in self.activations.items():
                # Check if dataset exists; if so, overwrite
                dataset_path = f"{model_name}/{prompt}/{key}"
                if dataset_path in f:
                    del f[dataset_path]  # Delete existing dataset
                f.create_dataset(dataset_path, data=act.cpu().numpy())

        print("Saving activations in H5 format.")
        with open(f"activations_{model_name}_{prompt.replace(' ', '_')}.pkl", "wb") as f:
            pickle.dump({key: act.cpu().numpy() for key, act in self.activations.items()}, f)

        self.activations.clear()
        print(f"Activations saved for {model_name}, prompt: {prompt}")

    def generate_answer_for_one_model(self, model_name, input_prompt, max_length=100, print_model_details=1, save_activations=True):
        if model_name not in self.models:
            raise ValueError(f"Model {model_name} is not loaded.")
        
        current_tokenizer = self.tokenizers[model_name]
        current_model = self.models[model_name]
        input_tokens = current_tokenizer(input_prompt, return_tensors="pt").to(self.device)
        output_tokens = current_model.generate(
            input_ids=input_tokens['input_ids'],
            attention_mask=input_tokens['attention_mask'],
            max_length=max_length,
            do_sample=True,
            pad_token_id=current_tokenizer.eos_token_id
        )

        if print_model_details:
            print("Model Details:", current_model)

        print(current_tokenizer.decode(output_tokens[0], skip_special_tokens=True))

        if self.activations and save_activations:  # Toggle saving
            self.save_activations(model_name, input_prompt)

        return current_tokenizer.decode(output_tokens[0], skip_special_tokens=True)

In [ ]:
# class ModelManagerExtreme:
#     def __init__(self):
#         self.models = {}
#         self.tokenizers = {}
#         self.device = "cuda" if torch.cuda.is_available() else "cpu"
#         self.activations = {}

#     def load_model(self, model_name, offload_folder="offload_weights"):
#         """Loads the model dynamically with offloading."""
#         if model_name not in self.models:
#             print(f"Loading model: {model_name}")
#             os.makedirs(offload_folder, exist_ok=True)
#             self.models[model_name] = AutoModelForCausalLM.from_pretrained(
#                 model_name,
#                 torch_dtype=torch.float16,
#                 device_map="auto",
#                 offload_folder=offload_folder,
#                 use_safetensors=True
#             )
#             self.tokenizers[model_name] = AutoTokenizer.from_pretrained(model_name)
#             print("Model loaded successfully!")
#         else:
#             print(f"Model {model_name} is already loaded.")

#     def register_hooks(self, model_name, layers_to_hook="all"):
#         """Registers hooks dynamically."""
#         if model_name not in self.models:
#             raise ValueError(f"Model {model_name} is not loaded.")
        
#         def hook_fn(name):
#             def hook(module, input, output):
#                 self.activations[name] = output[0].detach() if isinstance(output, tuple) else output.detach()
#             return hook

#         def detect_model_architecture(model):
#             if hasattr(model, "model"):
#                 layers = model.model.layers
#                 return layers, "up_proj", "down_proj", "self_attn", "mlp"
#             elif hasattr(model, "transformer"):
#                 layers = model.transformer.h
#                 return layers, "c_fc", "c_proj", "attn", "mlp"
#             else:
#                 raise ValueError("Unknown architecture")

#         model = self.models[model_name]
#         layers, ffn_in, ffn_out, attn_attr, ffn_module = detect_model_architecture(model)
#         start_layer = 0 if layers_to_hook == "all" else len(layers) - 3

#         for i, layer in enumerate(layers[start_layer:]):
#             getattr(layer, attn_attr).register_forward_hook(hook_fn(f"attn_layer{i}_{model_name}"))
#             getattr(getattr(layer, ffn_module), ffn_in).register_forward_hook(hook_fn(f"ffn_in_layer{i}_{model_name}"))
#             getattr(getattr(layer, ffn_module), ffn_out).register_forward_hook(hook_fn(f"ffn_out_layer{i}_{model_name}"))

#     def save_activations(self, model_name, prompt):
#         """Saves activations in both HDF5 and PKL formats."""
#         if not self.activations:
#             print(f"No activations to save for {model_name} with prompt: {prompt}")
#             return

#         with h5py.File("activations.h5", "a") as f:
#             for key, act in self.activations.items():
#                 f.create_dataset(f"{model_name}/{prompt}/{key}", data=act.cpu().numpy())

#         with open(f"activations_{model_name}_{prompt.replace(' ', '_')}.pkl", "wb") as f:
#             pickle.dump({key: act.cpu().numpy() for key, act in self.activations.items()}, f)

#         self.activations.clear()
#         print(f"Activations saved for {model_name}, prompt: {prompt}")

#     def generate_answer_for_one_model(self, model_name, input_prompt, max_length=100, print_model_details=1):
#         """Generates output and optionally saves activations."""
#         if model_name not in self.models:
#             raise ValueError(f"Model {model_name} is not loaded.")
        
#         current_tokenizer = self.tokenizers[model_name]
#         current_model = self.models[model_name]
#         input_tokens = current_tokenizer(input_prompt, return_tensors="pt").to(self.device)
#         output_tokens = current_model.generate(
#             input_ids=input_tokens['input_ids'],
#             attention_mask=input_tokens['attention_mask'],
#             max_length=max_length,
#             do_sample=True,
#             pad_token_id=current_tokenizer.eos_token_id
#         )

#         if print_model_details:
#             print("Model Details:", current_model)

#         if self.activations:
#             self.save_activations(model_name, input_prompt)

#         return current_tokenizer.decode(output_tokens[0], skip_special_tokens=True)


In [ ]:
model_list = ["deepseek-ai/deepseek-math-7b-instruct", "deepseek-ai/deepseek-coder-1.3b-base"]

manager = ModelManagerExtreme()
for name in model_list:
    manager.load_model(name)

for name in model_list:
    manager.register_hooks(name, layers_to_hook="all")

In [ ]:
problem = "What is 4 + 2 - 2?"
for name in model_list:
    answer = manager.generate_answer_for_one_model(name, problem, save_activations = False)
    print(answer)

In [ ]:
## Brainstorming Ideas

For a small paper


1. 2s problem


Don't let the parrot land the plane!

Even math-tuned models terrible at the problems.


Two sets of problems
1. Possible simple ones: "use 1, 2, 3, and 4 to make 10..."
2. Impossible ones
3. Different prompts:


## Save everything in a log



## --- OLD TESTS ----

## A mega list of interesting models

In [ ]:
# ### A mega list of models
# MEGA_MODEL_LIST = [
#     # Found using Grok 3
#     "gpt2",  # GPT-2 Small (117M)
#     "distilbert-base-uncased",  # DistilBERT (66M)
#     "TinyLLaMA",  # Placeholder; needs exact variant (e.g., "TinyLLaMA-50M-math")
#     "microsoft/phi-1",  # Phi-1 (125M)
#     "facebook/opt-66m",  # OPT-66M
#     "deepseek-ai/deepseek-coder-1.3b-base",  # DeepSeek-Coder-1.3B
#     "deepseek-ai/deepseek-math-7b-instruct",  # DeepSeek-Math-7B-Instruct
    
#     # Found using Copilot
#     "microsoft/phi-3-mini-4k-instruct",  # Phi-3 Mini (3.8B, 4K context)
#     "facebook/galactica-125m",  # Galactica-125M (close to my 120M suggestion)
#     "MathBERT",  # Ambiguous; assuming a custom/hypothetical checkpoint
#     "MathBERT-pretrained",  # Ambiguous; distinct from above?
#     "MetaMath-Mistral-7B",  # Likely "meta-math/MetaMath-Mixtral-7B" or similar
#     "WizardMath-7B",  # Likely "WizardLM/WizardMath-7B" or variant
#     "DeepSeek-R1",  # Ambiguous; might be "deepseek-ai/DeepSeek-R1"
#     "MAmmoTH-7B",  # Likely "TIGER-AI-Lab/MAmmoTH-7B" (math solver)
#     "google/gemma-3-4b-it",  # Gemma-3-4B (instruction-tuned, ~4B)
#     "deepseek-ai/DeepSeek-R1-Zero"  # Lightweight DeepSeek variant
# ]

## Model 1: Galactica-125m Model
Summary: It cannot even add 2 and 2!

In [ ]:
# Load the Galactica model and its tokenizer
model1_name = "facebook/galactica-125m"
tokenizer1 = AutoTokenizer.from_pretrained(model1_name)
model1 = AutoModelForCausalLM.from_pretrained(model1_name)

In [ ]:

# Get input text and tokenize
input_text = "What is 2+2?"
inputs = tokenizer1(input_text, return_tensors="pt")


# Generate output
outputs = model1.generate(
    input_ids=inputs['input_ids'],  # Required input IDs
    attention_mask=inputs['attention_mask'],  # Pass attention mask for longer inputs
    max_length=100,  # Limit the length of the response
    do_sample=True,  # Enable sampling for more creative responses
    pad_token_id=tokenizer.eos_token_id  # Handle padding correctly
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Model Response:", response)

# Model 2: Microsoft-phi1 Model

In [ ]:
# Load the Microsoft Phi1 model and its tokenizer
tokenizer2 = AutoTokenizer.from_pretrained("microsoft/phi-1")
model2 = AutoModelForCausalLM.from_pretrained("microsoft/phi-1", torch_dtype=torch.float16).to(device)

def get_answer(question: str) -> str:
    inputs = tokenizer2(question, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model2.generate(**inputs, max_new_tokens=10, do_sample=False, eos_token_id=tokenizer2.eos_token_id)
    answer_string = "Answer:" + tokenizer2.decode(outputs[0], skip_special_tokens=True)
    return(answer_string)

In [ ]:
# Test
print(get_answer("What is 2 + 2?"))
# print(get_answer("What is 2 + 2 - 2?"))

## Model manager before cleaning up

In [ ]:
# Class for managing multiple models
class ModelManager:
    def __init__(self):
        # Note: Each model would have its own tokenizer. Is that necessary? Experiments indicate yes.
        self.models = {}
        self.tokenizers = {}
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def load_model(self, model_name, offload_folder = "offload_weights", use_quantization = False):
        if model_name not in self.models:
            time_start = time.time()
            # Load model and tokenizer
            print(f"Loading model: {model_name}")

            # Way 1: All in GPU and if not available, CPU. No Splitting
            # self.models[model_name] = AutoModelForCausalLM.from_pretrained(model_name).to(self.device)
        
            # Way 2: Use "auto" device. 
                # Would try to use GPU. If memory not sufficient, will split and off-load to CPU
                # Possible con: Increased response time.
            
            if (use_quantization == False):
                print("Loading model WITHOUT QUANTIZATION.\n")
                self.models[model_name] = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    torch_dtype=torch.float16,
                    device_map="auto",  # Auto-offload to CPU/RAM if needed
                    offload_folder=offload_folder,
                    use_safetensors=True
                )

            # Way 3: Compress with Bits and Bytes.
            # Load with 8-bit or 4-bit quantization WITHOUT QLoRA

            else:
                print("Loading model WITH QUANTIZATION.\n")
                quantization_config = BitsAndBytesConfig(load_in_4bit=True) # Set load_in_4bit=True for 4-bit quantization
                self.models[model_name] = AutoModelForCausalLM.from_pretrained(
                    model_name,
                    quantization_config = quantization_config,
                    torch_dtype=torch.float16,
                    device_map="auto",  # Auto-offload to CPU/RAM if needed
                    offload_folder=offload_folder,
                    use_safetensors=True
                )
            
            self.tokenizers[model_name] = AutoTokenizer.from_pretrained(model_name)
            
            print("Model device:", self.models[model_name].device)
            print(f"\tModel loading time = {time.time() - time_start}")
        else:
            print(f"Model {model_name} is already loaded.")

    def generate_answer_for_one_model(self, model_name, input_prompt, max_length=8192, print_model_details = 1):
        print(f"Current model: {model_name}")
        if model_name not in self.models:
            raise ValueError(f"Model {model_name} is not loaded. Please load first using load_model(model_name).")
        
        time_start = time.time()
        current_tokenizer = self.tokenizers[model_name]
        current_model = self.models[model_name]

        # ------------- Tokenize Input ------------------

        # First simple way 
            # --> Did not work with Llama-4, possibly due to sharding (i.e., spreading out) model across devices with offloading.
            # makes all the more important to try bits and bytes. Unfortunately, my device can't compress very large models like Llama-4.
        print(self.device)
        input_tokens = current_tokenizer(input_prompt, return_tensors = "pt").to("cuda")
        
        # Second way: Manual matching of input tokens and attention masks across devices.

        # input_tokens = self.tokenizers[model_name](
        #                     input_prompt, 
        #                     return_tensors="pt",
        #                     padding="max_length",  # Explicit padding control
        #                     max_length=max_length  # Match model's max_position_embeddings
        #                 )
        # input_tokens["attention_mask"] = input_tokens["attention_mask"].bool()  # Convert to boolean
        # # ---> Manual device placement for attention mask
        # input_tokens["attention_mask"] = input_tokens["attention_mask"].to(
        #     self.models[model_name].device
        # )
    
        # # ---> Positional IDs generation
        # seq_len = input_tokens["input_ids"].shape[1]
        # position_ids = torch.arange(0, seq_len, dtype=torch.long).unsqueeze(0)
        
        # ----------- Generate Output ---------------

        output_tokens = current_model.generate(
                            # input_ids = input_tokens['input_ids'],  # Required input IDs
                            # attention_mask = input_tokens['attention_mask'],  # Pass attention mask for longer inputs
                            
                            **input_tokens,  # Automatic device alignment
                            # position_ids=position_ids,
                            # max_new_tokens = 100,
                            max_length = max_length,  # Limit the length of the response
                            do_sample = True,  # Enable sampling for more creative responses
                            pad_token_id = current_tokenizer.eos_token_id  # Handle padding correctly
                        )

        if (print_model_details):
            print("Model Details:\n" + "-"*10)
            print(current_model)
            print("\n" + "-"*100)
        output_to_return = current_tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        
        print(f"Time taken to generate reply = {time.time()- time_start}")
        return f"Reply: \n ------------------\n {output_to_return}\n [END of ANSWER]\n"
